# Standalone alignment transfer example

Run T2TT, attention, CTC and HuBERT, alignment diagnostics, five S2UT conditions, then inference and evaluation with the legacy trainer.

This notebook uses `--trainer legacy` and writes directly to Drive. See the [native workflow](../../docs/BENCHMARK_COMPARISON.md) for the main benchmark comparison and ZIP recovery.

Use a GPU runtime and run from the top. `UPDATES=2` is the default connectivity check. Use a new `ROOT` when changing experimental settings.

## GitHub access

For a private repository, create a fine-grained personal access token in GitHub **Settings → Developer settings → Personal access tokens → Fine-grained tokens**. Select the repository owner, choose **Only select repositories**, select `alignment-transfer-s2ut`, and grant **Contents → Read-only**. Set an expiration date. See the [GitHub token guide](https://docs.github.com/en/authentication/keeping-your-account-and-data-secure/managing-your-personal-access-tokens).

In Colab, open **Secrets** (the key icon), add `GITHUB_TOKEN`, paste the token into its value, and enable **Notebook access**. The setup cell reads this secret for Git operations. Public repositories also work without a token. Successful clone/fetch and checkout in the setup cell confirm access to the requested revision.

To authenticate an additional repository, add its exact HTTPS URL to `GITHUB_AUTH_REPOS` in the settings cell and grant the same token read access to that repository. The benchmark is public and uses anonymous HTTPS by default.

After a runtime reset, use the same notebook revision, experiment settings, and Drive paths, then rerun the cells. To open the notebook after the repository becomes private, download the `.ipynb` while signed into GitHub and use **File → Upload notebook** in Colab, or open a copy saved in your Drive. Enable secret access for that notebook. The runtime token is used after the notebook opens.


## 1. Mount Drive and configure the run

In [ ]:
#@title Process logging and progress
from pathlib import Path
import json
"""Standard-library-only process logging, also embedded in Colab before install."""


def log_event(name, state, logs, detail=""):
    from datetime import datetime, timezone
    from pathlib import Path
    root = Path(logs)
    root.mkdir(parents=True, exist_ok=True)
    line = f"{datetime.now(timezone.utc).isoformat()} [{name}] {state} {detail}\n"
    with (root / "notebook-events.log").open("a", encoding="utf-8") as stream:
        stream.write(line)
        stream.flush()
    print(line, end="", flush=True)


def graceful_sigterm(function):
    """Turn a first POSIX SIGTERM into the ordinary interruption cleanup path."""
    from functools import wraps
    import os
    import signal
    import threading

    @wraps(function)
    def wrapped(*args, **kwargs):
        if (os.name != "posix" or threading.current_thread() is not threading.main_thread()
                or signal.getsignal(signal.SIGTERM) != signal.SIG_DFL):
            return function(*args, **kwargs)
        interrupted = False

        def interrupt(signum, frame):
            nonlocal interrupted
            if not interrupted:
                interrupted = True
                raise KeyboardInterrupt("SIGTERM")

        previous = signal.signal(signal.SIGTERM, interrupt)
        try:
            return function(*args, **kwargs)
        finally:
            signal.signal(signal.SIGTERM, previous)
    return wrapped


@graceful_sigterm
def run_process(command, logs, *, name="command", capture_json=False, env=None, cwd=None,
                heartbeat_seconds=10., extra_record=None):
    """Stream every child line; silence triggers liveness, never fake progress.

    stdout JSON is kept apart from stderr. Nested loggers share an owned POSIX
    process group so interruption cannot strand a second training process.
    """
    from datetime import datetime, timezone
    import hashlib
    import json
    import os
    from pathlib import Path
    import queue
    import signal
    import subprocess
    import tempfile
    import threading
    import time

    if heartbeat_seconds <= 0:
        raise ValueError("heartbeat_seconds must be positive")
    root = Path(logs)
    root.mkdir(parents=True, exist_ok=True)
    command = list(map(str, command))
    key = hashlib.sha256(json.dumps(name, ensure_ascii=False).encode()).hexdigest()[:12] + "-" + str(time.time_ns())
    output, result_path, receipt = root / (key + ".log"), root / (key + ".stdout.log"), root / (key + ".json")
    launch_env = dict(os.environ if env is None else env)
    launch_env["PYTHONUNBUFFERED"] = "1"
    termination_timeout = 75 if launch_env.get("AT_LOG_WANDB_DRAIN") == "1" else 5
    owns_group = os.name == "posix" and launch_env.get("AT_LOG_MANAGED_GROUP") != "1"
    if owns_group:
        launch_env["AT_LOG_MANAGED_GROUP"] = "1"
    started, clock = datetime.now(timezone.utc).isoformat(), time.perf_counter()
    record = dict(name=name, command=command, started_utc=started,
        host_utc_offset_seconds=int(datetime.now().astimezone().utcoffset().total_seconds()),
        elapsed_seconds=0., returncode=None, status="running", exception=None,
        pid=None, log=str(output.resolve()), stdout_log=str(result_path.resolve()) if capture_json else None,
        gpu_csv=None, gpu_monitor_error=None, cwd=str(Path(cwd or Path.cwd()).resolve()))
    lock, stop, done = threading.Lock(), threading.Event(), threading.Event()
    messages, reader_errors = queue.Queue(maxsize=256), []
    process = reader = result_stream = None
    stream = output.open("w", encoding="utf-8")

    def save_record():
        record["elapsed_seconds"] = time.perf_counter() - clock
        fd, temporary = tempfile.mkstemp(dir=root, prefix=".log-record-")
        try:
            with os.fdopen(fd, "w", encoding="utf-8") as target:
                json.dump(record, target, ensure_ascii=False, indent=2, allow_nan=False)
                target.write("\n")
                target.flush()
            os.replace(temporary, receipt)
        finally:
            if os.path.exists(temporary):
                os.unlink(temporary)

    def notice(state, detail=""):
        line = f"[{name}] {state} pid={record['pid']} elapsed={time.perf_counter()-clock:.1f}s {detail}\n"
        with lock:
            stream.write(line)
            stream.flush()
        print(line, end="", flush=True)

    def read_lines(pipe):
        try:
            for line in pipe:
                with lock:
                    stream.write(line)
                    stream.flush()
                while not stop.is_set():
                    try:
                        messages.put(line, timeout=.1)
                        break
                    except queue.Full:
                        pass
        except Exception as error:
            reader_errors.append(error)
        finally:
            done.set()

    def terminate_child():
        if process is None:
            return
        if owns_group:
            try:
                os.killpg(process.pid, signal.SIGTERM)
            except ProcessLookupError:
                pass
        elif process.poll() is None:
            process.terminate()
        try:
            process.wait(timeout=termination_timeout)
        except subprocess.TimeoutExpired:
            if owns_group:
                os.killpg(process.pid, signal.SIGKILL)
            else:
                process.kill()
            process.wait()
        if owns_group:
            # A nested child may outlive its exited leader after cancellation.
            try:
                os.killpg(process.pid, signal.SIGKILL)
            except ProcessLookupError:
                pass

    try:
        notice("started", f"full log: {output.resolve()}")
        save_record()
        if capture_json:
            result_stream = result_path.open("w", encoding="utf-8")
        process = subprocess.Popen(command, stdout=result_stream if capture_json else subprocess.PIPE,
            stderr=subprocess.PIPE if capture_json else subprocess.STDOUT, text=True,
            encoding="utf-8", errors="replace", bufsize=1, env=launch_env, cwd=cwd,
            start_new_session=owns_group)
        record["pid"] = process.pid
        notice("running", "waiting for child output")
        save_record()
        pipe = process.stderr if capture_json else process.stdout
        reader = threading.Thread(target=read_lines, args=(pipe,), daemon=True, name="command-log-reader")
        reader.start()
        last_output = last_notice = time.perf_counter()
        while not (done.is_set() and messages.empty() and process.poll() is not None):
            try:
                line = messages.get(timeout=min(.1, heartbeat_seconds))
            except queue.Empty:
                line = None
            if line is not None:
                print(line, end="", flush=True)
                if line.strip():
                    last_output = time.perf_counter()
            if reader_errors:
                raise reader_errors[0]
            now = time.perf_counter()
            if now - max(last_output, last_notice) >= heartbeat_seconds:
                notice("running", f"no_output={now-last_output:.1f}s (liveness only; progress not confirmed)")
                save_record()
                last_notice = now
        code = process.wait()
        if reader_errors:
            raise reader_errors[0]
        if capture_json and code == 0:
            json.loads(result_path.read_text(encoding="utf-8"))
        record.update(returncode=code, status="succeeded" if code == 0 else "failed_or_interrupted")
        if code:
            raise subprocess.CalledProcessError(code, command)
        notice("completed", f"exit={code}; full log: {output.resolve()}")
    except BaseException as error:
        record.update(status="failed_or_interrupted", exception=type(error).__name__ + ": " + str(error))
        terminate_child()
        if process is not None:
            record["returncode"] = process.returncode
        notice("interrupted" if isinstance(error, KeyboardInterrupt) else "failed",
               f"exit={record['returncode']} {type(error).__name__}; full log: {output.resolve()}")
        raise
    finally:
        stop.set()
        if reader is not None:
            reader.join(timeout=2)
        if process is not None and (reader is None or not reader.is_alive()):
            pipe = process.stderr if capture_json else process.stdout
            if pipe is not None:
                pipe.close()
        if result_stream is not None:
            result_stream.close()
        if extra_record is not None:
            record.update(extra_record() if callable(extra_record) else extra_record)
        save_record()
        stream.close()
    result = subprocess.CompletedProcess(command, record["returncode"],
        result_path.read_text(encoding="utf-8") if capture_json else None)
    result.record = record
    return result


"""Standard-library GitHub authentication, embedded before Colab installation."""


GITHUB_GIT_WORKER = r'''
import base64
import os
import subprocess
import sys
from urllib.parse import quote

redactions = []
for i in range(int(os.environ.get("GIT_CONFIG_COUNT", "0"))):
    value = os.environ.get(f"GIT_CONFIG_VALUE_{i}", "")
    if value.startswith("Authorization: Basic "):
        encoded = value.split()[-1]
        token = base64.b64decode(encoded).decode("utf-8").split(":", 1)[1]
        redactions.extend((value, encoded, token, quote(token, safe="")))
redactions = sorted(set(redactions), key=len, reverse=True)
process = None
try:
    command = sys.argv[1:]
    position = 1
    while position < len(command) and command[position] == "-C":
        position += 2
    if command[position] in {"fetch", "pull"}:
        remote = next((arg for arg in command[position + 1:] if not arg.startswith("-")), "origin")
        if "://" not in remote:
            checked = subprocess.run([*command[:position], "remote", "get-url", "--all", remote],
                stdin=subprocess.DEVNULL, stdout=subprocess.PIPE, stderr=subprocess.PIPE,
                text=True, encoding="utf-8", errors="replace")
            from urllib.parse import urlsplit
            for address in checked.stdout.splitlines():
                url = urlsplit(address)
                if url.username or url.password or url.query or url.fragment:
                    raise ValueError("unsafe stored remote")
    process = subprocess.Popen(command, stdin=subprocess.DEVNULL,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
        encoding="utf-8", errors="replace", bufsize=1)
    for line in process.stdout:
        for secret in redactions:
            line = line.replace(secret, "[REDACTED]")
        print(line, end="", flush=True)
    code = process.wait()
except Exception:
    print("Git could not complete the requested operation.", file=sys.stderr, flush=True)
    code = 1
finally:
    if process is not None:
        if process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
        process.stdout.close()
if code:
    print("Check the repository URL and revision. For a private repository, add "
          "GITHUB_TOKEN to Colab Secrets, enable notebook access, and grant the "
          "fine-grained token Contents: Read-only for the selected repository. "
          "Check token expiry and any organization approval.", file=sys.stderr, flush=True)
sys.exit(code)
'''


def run_github_git(command, logs, *, runner, repositories, name="git", env=None, cwd=None,
                   capture_json=False):
    """Authenticate network Git calls with URL-scoped, process-only headers."""
    import base64
    import os
    from pathlib import Path
    import re
    import sys
    from urllib.parse import urlsplit

    command = list(map(str, command))
    position = 1
    while position < len(command) and command[position] == "-C":
        position += 2
    operation = command[position] if position < len(command) else ""
    if operation not in {"clone", "fetch", "pull", "ls-remote"}:
        return runner(command, logs, name=name, env=env, cwd=cwd, capture_json=capture_json)
    if Path(command[0]).name.lower() not in {"git", "git.exe"}:
        raise ValueError("GitHub authentication requires a git command")
    for argument in command:
        if "://" in argument:
            url = urlsplit(argument)
            if url.username or url.password or url.query or url.fragment:
                raise ValueError("Use repository URLs without credentials, queries, or fragments")
    scopes = []
    for repository in repositories:
        url = urlsplit(str(repository))
        if (url.scheme != "https" or url.netloc != "github.com" or url.query or url.fragment
                or not re.fullmatch(r"/[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+", url.path)):
            raise ValueError("Authentication scopes must be plain HTTPS GitHub repository URLs")
        scopes.append(url.geturl())

    token = None
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        pass  # Public repositories remain accessible when Secrets are unavailable.
    if token is not None and (not isinstance(token, str) or any(ord(c) < 33 or ord(c) > 126 for c in token)):
        token = None
        raise ValueError("GITHUB_TOKEN must contain a valid GitHub token without whitespace")
    git_env = dict(os.environ if env is None else env)
    for key in list(git_env):
        if (key.startswith(("GIT_TRACE", "GIT_CONFIG_KEY_", "GIT_CONFIG_VALUE_"))
                or key in {"GIT_CONFIG_COUNT", "GIT_CONFIG_PARAMETERS", "GIT_CURL_VERBOSE", "GITHUB_TOKEN", "GH_TOKEN"}):
            git_env.pop(key)
    git_env.update(GIT_TERMINAL_PROMPT="0", GCM_INTERACTIVE="Never", GIT_ASKPASS="", GIT_TRACE_REDACT="1")
    settings = [("credential.helper", ""), ("http.extraHeader", ""), ("http.followRedirects", "false")]
    header = None
    try:
        if token:
            header = "Authorization: Basic " + base64.b64encode(("x-access-token:" + token).encode()).decode()
            settings.extend(("http." + scope + ".extraHeader", header) for scope in dict.fromkeys(scopes))
        git_env["GIT_CONFIG_COUNT"] = str(len(settings))
        for i, (key, value) in enumerate(settings):
            git_env[f"GIT_CONFIG_KEY_{i}"] = key
            git_env[f"GIT_CONFIG_VALUE_{i}"] = value
        return runner([sys.executable, "-c", GITHUB_GIT_WORKER, *command], logs,
                      name=name, env=git_env, cwd=cwd, capture_json=capture_json)
    except Exception:
        raise RuntimeError("GitHub access failed. Check the logged URL/revision error and Colab Secrets "
                           "GITHUB_TOKEN access, selected repositories, Contents: Read-only, and expiry.") from None
    finally:
        token = header = None
        settings.clear()
        git_env.clear()


def execute(*arguments, name=None, capture_json=False, env=None, cwd=None):
    command = list(map(str, arguments))
    if name is None:
        name = command[2].rsplit('.', 1)[-1] if len(command) > 2 and command[1] == '-m' else Path(command[0]).name
        if '--method' in command:
            name += ':' + command[command.index('--method') + 1]
        if '--prior-type' in command:
            name += ':' + command[command.index('--prior-type') + 1]
    if command and command[0] == 'git':
        repositories = [globals().get('REPO_URL', 'https://github.com/Yaaamashiro/alignment-transfer-s2ut.git'),
                        *globals().get('GITHUB_AUTH_REPOS', [])]
        return run_github_git(command, LOGS, runner=run_process, repositories=repositories,
                              name=name, capture_json=capture_json, env=env, cwd=cwd)
    return run_process(command, LOGS, name=name, capture_json=capture_json, env=env, cwd=cwd)


def execute_json(*arguments, name=None, env=None, cwd=None):
    return json.loads(execute(*arguments, name=name, capture_json=True, env=env, cwd=cwd).stdout)

LOGS = Path('/content/alignment-transfer-logs/pipeline/bootstrap')


In [ ]:
from google.colab import drive
log_event('drive-mount', 'started', LOGS)
try:
    drive.mount('/content/drive')
except BaseException as error:
    log_event('drive-mount', 'failed', LOGS, type(error).__name__)
    raise
else:
    log_event('drive-mount', 'completed', LOGS)
import os
REPO_URL = 'https://github.com/Yaaamashiro/alignment-transfer-s2ut.git' #@param {type:"string"}
REPO_REVISION = '107605f4f5bd7b53722d73ced3fa5a76560e7a9d' #@param {type:"string"}
ROOT = '/content/drive/MyDrive/alignment-transfer-s2ut-data/experiment-stream-001' #@param {type:"string"}
CORPUS = '/content/drive/MyDrive/ja-en-direct-s2st-corpus-data' #@param {type:"string"}
RECIPE = 'controlled' #@param ['controlled', 'strong']
RUNS = ROOT + '/' + RECIPE
ANNEAL_STEPS = 10000 #@param {type:"integer"}
UPDATES = 2 #@param {type:"integer"}
PYTHON = '/content/alignment-venv/bin/python'
LOGS = Path('/content/alignment-transfer-logs/pipeline') / Path(ROOT).name / RECIPE
os.environ['HF_HOME'] = '/content/huggingface'
os.environ['MPLBACKEND'] = 'Agg'
os.environ['PYTHONUNBUFFERED'] = '1'


## 2. Set up and check the environment

Commands record startup, PID, live output, and completion or failure in `LOGS`. After ten seconds of silence, the logger reports liveness.


In [ ]:
# Clone/install only. Never point outputs at the corpus or benchmark repositories.
import subprocess, sys

from pathlib import Path
assert len(REPO_REVISION) == 40 and 'SET_' not in REPO_URL
assert not Path(ROOT).resolve().is_relative_to(Path(CORPUS).resolve())
if not Path('/content/alignment-transfer-s2ut/.git').exists():
    execute('git', 'clone', '--no-checkout', REPO_URL, '/content/alignment-transfer-s2ut')
execute('git', '-C', '/content/alignment-transfer-s2ut', 'fetch', 'origin', REPO_REVISION)
execute('git', '-C', '/content/alignment-transfer-s2ut', 'checkout', '--detach', REPO_REVISION)
os.chdir('/content/alignment-transfer-s2ut')
execute(
    sys.executable, '-m', 'pip', 'install', 'uv'
)
execute(
    'uv', 'venv', '--python', '3.10.22', '--seed', '--allow-existing', '/content/alignment-venv'
)
execute(
    PYTHON, '-m', 'pip', 'install', 'pip==23.3.2', 'setuptools<81', 'wheel'
)
execute(
    PYTHON, '-m', 'pip', 'install', 'torch==2.7.1', 'torchaudio==2.7.1', 'numpy==1.26.4',
    'Cython==3.2.9'
)
execute(
    PYTHON, '-m', 'pip', 'install', '--no-build-isolation', '-r', 'requirements-fairseq.txt'
)
execute(
    PYTHON, '-m', 'pip', 'install', '-r', 'requirements.txt'
)
execute(
    PYTHON, '-m', 'pytest', '-v', '--tb=short'
)
execute(PYTHON, '-m', 'alignment_transfer_s2ut.pilot', 'runtime')
execute(PYTHON, '-m', 'pip', 'check', name='dependency-check')


## 3. Verify the manifest and shared settings

In [ ]:
# First import only. On resume, retain the existing canonical manifest unchanged.
execute(
    PYTHON, '-m', 'alignment_transfer_s2ut.prepare_manifest', '--input',
    f'{CORPUS}/production/manifests/releases/accepted.jsonl', '--corpus-root', f'{CORPUS}',
    '--output', f'{ROOT}/manifest.jsonl', '--resume'
)
execute(
    PYTHON, '-m', 'alignment_transfer_s2ut.compare_architectures', '--t2tt-config',
    'configs/t2tt/fisher.yaml', '--s2ut-config', 'configs/s2ut/fisher.yaml', '--output',
    f'{ROOT}/architecture.json', '--resume'
)


## 4. Train T2TT

In [ ]:
# Keep the same recipe; resume automatically if a checkpoint exists.
execute(
    PYTHON, '-m', 'alignment_transfer_s2ut.train_t2tt', '--manifest', f'{ROOT}/manifest.jsonl',
    '--config', 'configs/t2tt/fisher.yaml', '--output', f'{ROOT}/t2tt', '--max-updates',
    f'{UPDATES}', '--device', 'cuda', '--resume-if-exists'
)


## 5. Build attention, CTC, units, and the teacher

In [ ]:
execute(
    PYTHON, '-m', 'alignment_transfer_s2ut.extract_t2tt_attention', '--manifest',
    f'{ROOT}/manifest.jsonl', '--checkpoint', f'{ROOT}/t2tt/checkpoint_best.pt', '--output',
    f'{ROOT}/mt', '--device', 'cuda', '--resume'
)
execute(
    PYTHON, '-m', 'alignment_transfer_s2ut.align_speech_text', '--manifest',
    f'{ROOT}/manifest.jsonl', '--config', 'configs/alignment/ctc.yaml', '--output', f'{ROOT}/ctc',
    '--device', 'cuda', '--resume'
)
execute(
    PYTHON, '-m', 'alignment_transfer_s2ut.fetch_assets', '--config',
    'configs/units/hubert_km100.yaml'
)
execute(
    PYTHON, '-m', 'alignment_transfer_s2ut.extract_units', '--manifest', f'{ROOT}/manifest.jsonl',
    '--config', 'configs/units/hubert_km100.yaml', '--output', f'{ROOT}/units', '--device', 'cuda',
    '--resume'
)
execute(
    PYTHON, '-m', 'alignment_transfer_s2ut.build_teacher_alignment', '--manifest',
    f'{ROOT}/manifest.jsonl', '--mt-attention', f'{ROOT}/mt', '--alignment', f'{ROOT}/ctc',
    '--units', f'{ROOT}/units', '--config', 'configs/s2ut/fisher.yaml', '--output',
    f'{ROOT}/teacher', '--resume'
)


## 6. Alignment diagnostics

In [ ]:
# Audit coverage/uncertainty before training; inspect heatmaps against original audio.
execute(
    PYTHON, '-m', 'alignment_transfer_s2ut.diagnose_alignment', '--manifest',
    f'{ROOT}/manifest.jsonl', '--teacher', f'{ROOT}/teacher', '--units', f'{ROOT}/units',
    '--config', 'configs/s2ut/fisher.yaml', '--split', 'dev', '--max-samples', '500', '--heatmaps',
    '20', '--source-mapping', 'frontend', '--prior-smoothing', '0.05', '--output',
    f'{ROOT}/alignment-audit', '--overwrite'
)


Choose `controlled` or `strong` with `RECIPE`. Strong uses auxiliary character decoders, decoder CTC, CMVN, and SpecAugment. Both include a monotonic prior control and use separate output directories. Use `UPDATES=2` for connectivity, 5000–20000 for a pilot, or a common budget across conditions for the main experiment. Incompatible resume contracts require a new `ROOT`.

## 7. Train five S2UT conditions

In [ ]:
# One recipe, timing and smoothing across all five conditions.
SHARED = ['--trainer', 'legacy', '--manifest', f'{ROOT}/manifest.jsonl',
    '--config', 'configs/s2ut/fisher.yaml', '--units', f'{ROOT}/units',
    '--loader-workers', 2, '--prefetch-batches', 2, '--cache-dir', '/content/alignment-transfer-cache',
    '--cache-gb', 8, '--pin-memory', '--recipe', RECIPE, '--prior-smoothing', 0.05,
    '--source-mapping', 'frontend', '--anneal-steps', ANNEAL_STEPS,
    '--validation-interval', 1000, '--save-interval-updates', 1000,
    '--max-updates', UPDATES, '--device', 'cuda', '--resume-if-exists']
CONDITIONS = [('baseline', 'baseline', 't2tt'), ('distillation', 'distillation', 't2tt'),
    ('guidance', 'guidance', 't2tt'), ('annealed_guidance', 'annealed_guidance', 't2tt'),
    ('annealed_monotonic', 'annealed_guidance', 'monotonic')]
for name, method, prior in CONDITIONS:
    teacher = ['--teacher', f'{ROOT}/teacher'] if name not in ('baseline', 'annealed_monotonic') else []
    control = ['--prior-type', 'monotonic'] if prior == 'monotonic' else []
    execute(PYTHON, '-m', 'alignment_transfer_s2ut.train_s2ut', *SHARED,
        '--method', method, '--output', f'{RUNS}/{name}', *teacher, *control)
execute(PYTHON, '-m', 'alignment_transfer_s2ut.compare_experiments',
    *[f'{RUNS}/{name}' for name, _, _ in CONDITIONS])


Prepare source-only JSONL with `pair_id` and `ja_audio`. Match the split and ID set used for evaluation.

## 8. Inference and final evaluation

Use dev to select training settings and reserve test for final evaluation. Two-update runs check connectivity.

In [ ]:
SPEECH_ONLY = ROOT + '/test_speech_only.jsonl' #@param {type:"string"}
METHOD = 'annealed_guidance' #@param ['baseline', 'distillation', 'guidance', 'annealed_guidance', 'annealed_monotonic']
execute(
    PYTHON, '-m', 'alignment_transfer_s2ut.prepare_inference_input', '--manifest',
    f'{ROOT}/manifest.jsonl', '--split', 'test', '--output', f'{SPEECH_ONLY}', '--resume'
)
execute(
    PYTHON, '-m', 'alignment_transfer_s2ut.fetch_assets', '--config', 'configs/s2ut/vocoder.yaml'
)
execute(
    PYTHON, '-m', 'alignment_transfer_s2ut.infer', '--input', f'{SPEECH_ONLY}', '--checkpoint',
    f'{RUNS}/{METHOD}/checkpoint_best.pt', '--vocoder-config', 'configs/s2ut/vocoder.yaml',
    '--output', f'{RUNS}/{METHOD}/test-predictions.jsonl', '--device', 'cuda', '--resume'
)
execute(
    PYTHON, '-m', 'alignment_transfer_s2ut.evaluate', '--manifest', f'{ROOT}/manifest.jsonl',
    '--split', 'test', '--predictions', f'{RUNS}/{METHOD}/test-predictions.jsonl', '--units',
    f'{ROOT}/units', '--asr-config', 'configs/experiments/evaluation.yaml', '--output',
    f'{RUNS}/{METHOD}/test-metrics.json', '--device', 'cuda', '--resume'
)


## Outputs and recovery

Save the text teacher in `ROOT/t2tt/`, projected alignment in `ROOT/teacher/`, and condition results under `ROOT/<recipe>/<condition>/`. Commands stop on failure. Rerun with the same settings and `ROOT` to reuse completed artifacts and checkpoints.

The legacy workflow writes checkpoints directly to Drive. The main benchmark workflow uses local work and verified ZIPs.